# QLoRA for Intent Classification (Banking77)

## Overview

This notebook applies **QLoRA** (Quantized Low-Rank Adaptation) to **intent classification** on Banking77, using a 4-bit quantized causal LLM (**Qwen3-4B-Instruct-2507**).

| # | Approach | Question |
|---|---|---|
| 0 | **TF-IDF + logistic regression** | How far does a cheap, non-neural baseline get? |
| 1 | **Instruction prompting** | Given the 77 intent names, can the frozen model pick the right one? |
| 2 | **Chain-of-Thought prompting** | Does reasoning first improve accuracy, and at what token cost? |
| 2b | **Retrieved few-shot prompting** (optional) | Do nearest-neighbour examples in the prompt close the gap? |
| 3 | **QLoRA fine-tuning** | How much does training ~1% of parameters on ~1.5k labelled queries help? |

### Why Banking77?

Banking77 contains 13,083 short customer-support queries labelled with **77 fine-grained intents**. The task is simple to state but hard to solve zero-shot:

* **The label set is a convention.** Intent boundaries such as `card_arrival` vs `card_delivery_estimate` are defined by the dataset, not by general language knowledge.
* **Many intents overlap.** Pairs like `lost_or_stolen_card` / `compromised_card` or `top_up_failed` / `top_up_reverted` differ in details a model has to learn from examples.
* **Output validity matters.** A generative model must emit one of exactly 77 strings; anything else is an error.

### Model

We load `Qwen/Qwen3-4B-Instruct-2507` and quantize it on load to 4-bit NF4 with double quantization (Dettmers et al., 2023). QLoRA then trains small LoRA adapters on top of the frozen quantized weights. Peak GPU memory stays within a free Colab T4 (16 GB).

### Metrics

Every method is scored on the **same stratified test sample**:

* **Accuracy** with a 95% bootstrap confidence interval
* **Macro-F1** over the 77 intents
* **Invalid rate**: outputs that cannot be mapped to a valid intent
* **Generated tokens per query** and **latency per query** (LLM methods)

Predictions are computed once, cached in `RESULTS`, and reused by every table and plot.

## Setup

### Dependencies

!pip install -q -U transformers datasets peft accelerate bitsandbytes scikit-learn pandas matplotlib

### Imports, seeds and device

In [ ]:
import re, gc, json, time, random, difflib, collections, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import datasets
import transformers
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, TrainingArguments, Trainer)
from peft import LoraConfig, TaskType, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
transformers.set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU:  {props.name}")
    print(f"  VRAM: {props.total_memory / 1e9:.1f} GB")

# bf16 needs Ampere (compute capability 8.x) or newer. A T4 is 7.5, so it falls back to fp16.
# torch.cuda.is_bf16_supported() can return True on a T4 through emulation, so we check the
# compute capability directly.
USE_BF16 = device == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Compute dtype: {DTYPE}")

### Configuration

All knobs live here. The defaults are sized for a free Colab T4.

`TRAIN_PROMPT_STYLE` controls the prompt used for fine-tuning and for evaluating the fine-tuned model:

* `"full"` keeps the 77 intent names in the system prompt, exactly like the zero-shot prompt.
* `"minimal"` drops the list. A fine-tuned model learns the label set from the data, so the list is no longer needed, and each sequence becomes roughly 10x shorter, which makes training and inference much faster.

In [4]:
MODEL_ID   = "Qwen/Qwen3-4B-Instruct-2507"
DATASET_ID = "PolyAI/banking77"

# Data sizes (all samples are stratified by intent)
TRAIN_SIZE = 1540     # about 20 queries per intent on average
VAL_SIZE   = 308      # used only for eval loss during training
TEST_SIZE  = 500      # the same sample is used for every method

# Inference
GEN_BATCH_SIZE = 16
RUN_FEW_SHOT   = True
FEW_SHOT_K     = 8

# QLoRA
TRAIN_PROMPT_STYLE = "minimal"   # "minimal" or "full"
RANK       = 16
EPOCHS     = 2
LR         = 2e-4
OUTPUT_DIR  = "./qlora_banking77"
ADAPTER_DIR = "./qlora_banking77_adapter"

### Load the model in 4-bit

`BitsAndBytesConfig` is the "Q" in QLoRA: weights are stored as 4-bit NF4, the quantization constants are themselves quantized (double quantization), and matmuls run in `DTYPE`.

Note on parameter counts: bitsandbytes packs two 4-bit values into each stored element, so `numel()` on a quantized weight reports half the true count. `count_parameters` corrects for this.

In [ ]:
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

def load_base_model():
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=BNB_CONFIG,
        dtype=DTYPE,
        device_map="auto",
    )

def count_parameters(m):
    total = trainable = 0
    for p in m.parameters():
        n = p.numel() * (2 if p.__class__.__name__ == "Params4bit" else 1)
        total += n
        if p.requires_grad:
            trainable += n
    return total, trainable

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"            # required for batched generation with decoder-only models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = load_base_model()

total, trainable = count_parameters(model)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}  ({100 * trainable / total:.2f}%, pre-LoRA)")
if device == "cuda":
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Dataset: Banking77

**HuggingFace identifier:** `PolyAI/banking77`
**Paper:** Casanueva et al. (2020), *"Efficient Intent Detection with Dual Sentence Encoders"*.
**Licence:** CC BY 4.0.

| Split | Size |
|---|---|
| train | 10,003 |
| test | 3,080 |

Each example has a `text` (one customer query) and a `label` (one of 77 intents). The classes are imbalanced, so every subset below is drawn **stratified by intent**: a random sample of 500 would leave some intents with almost no test queries.

A validation split is carved out of train (10%), and the test sample is touched only for the final scores of each method.

In [ ]:
try:
    raw = datasets.load_dataset(DATASET_ID)
except Exception as e:
    print(f"Default load failed ({type(e).__name__}); using the parquet conversion branch.")
    raw = datasets.load_dataset(DATASET_ID, revision="refs/convert/parquet")
print(raw)

LABELS   = raw["train"].features["label"].names
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = dict(enumerate(LABELS))
NUM_LABELS = len(LABELS)
print(f"\n{NUM_LABELS} intents, e.g. {LABELS[:5]}")

In [ ]:
def stratified_indices(labels, n, seed=SEED):
    # Indices of a stratified sample of size n (all indices if n >= len(labels)).
    labels = np.asarray(labels)
    idx = np.arange(len(labels))
    if n >= len(labels):
        return idx
    _, keep = train_test_split(idx, test_size=n, stratify=labels, random_state=seed)
    return np.sort(keep)

train_labels_all = np.asarray(list(raw["train"]["label"]))
pool_idx, val_pool_idx = train_test_split(
    np.arange(len(train_labels_all)), test_size=0.1,
    stratify=train_labels_all, random_state=SEED,
)
train_pool = raw["train"].select(pool_idx)       # ~9k queries, used only by the "all data" TF-IDF reference
val_pool   = raw["train"].select(val_pool_idx)

train_ds = train_pool.select(stratified_indices(list(train_pool["label"]), TRAIN_SIZE)).shuffle(seed=SEED)
val_ds   = val_pool.select(stratified_indices(list(val_pool["label"]), VAL_SIZE))
test_ds  = raw["test"].select(stratified_indices(list(raw["test"]["label"]), TEST_SIZE))

TEST_TEXTS = list(test_ds["text"])
TEST_GOLD  = [id2label[l] for l in test_ds["label"]]

per_class = collections.Counter(train_ds["label"])
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Train queries per intent: min {min(per_class.values())}, "
      f"median {int(np.median(list(per_class.values())))}, max {max(per_class.values())}")
print(f"Intents present in test sample: {len(set(TEST_GOLD))} / {NUM_LABELS}")

### Inspect the data

In [ ]:
for ex in raw["train"].shuffle(seed=SEED).select(range(10)):
    print(f"{id2label[ex['label']]:<40} {ex['text']}")

lens = [len(t.split()) for t in raw["train"]["text"]]
print(f"\nQuery length (words): min {min(lens)}, mean {np.mean(lens):.1f}, max {max(lens)}")

In [ ]:
counts = collections.Counter(train_labels_all)
order = sorted(range(NUM_LABELS), key=lambda i: counts[i], reverse=True)

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.bar(range(NUM_LABELS), [counts[i] for i in order])
ax.set_xticks(range(NUM_LABELS))
ax.set_xticklabels([LABELS[i] for i in order], rotation=90, fontsize=6)
ax.set_ylabel("train queries")
ax.set_title("Banking77 class distribution (full train split)")
plt.tight_layout()
plt.show()

print(f"Majority class: {LABELS[order[0]]} ({counts[order[0]]} queries, "
      f"{100 * counts[order[0]] / len(train_labels_all):.1f}% of train)")
print(f"Random-guess accuracy: {100 / NUM_LABELS:.1f}%")

Some intents are close neighbours. One query from each side of a few such pairs shows how small the distinctions can be:

In [ ]:
CONFUSABLE_PAIRS = [
    ("card_arrival", "card_delivery_estimate"),
    ("lost_or_stolen_card", "compromised_card"),
    ("top_up_failed", "top_up_reverted"),
    ("pending_transfer", "transfer_not_received_by_recipient"),
]
by_label = collections.defaultdict(list)
for ex in raw["train"]:
    by_label[id2label[ex["label"]]].append(ex["text"])

for a, b in CONFUSABLE_PAIRS:
    if a in label2id and b in label2id:
        print(f"{a:<38} {by_label[a][0]}")
        print(f"{b:<38} {by_label[b][0]}\n")

## Prompts and output parsing

Three system prompts:

* **Instruction**: list the 77 intents and ask for the intent name only.
* **Chain-of-Thought**: same list, but reason briefly first and finish with `Label: <intent>`.
* **Minimal**: no list. Only used for the fine-tuned model when `TRAIN_PROMPT_STYLE = "minimal"`.

The label list is the dominant cost of the zero-shot prompts, so we measure it.

In [ ]:
LABEL_LIST = "\n".join(LABELS)

SYSTEM_INSTRUCTION = (
    "You classify messages sent by customers to a bank's support channel into exactly one intent.\n"
    "Valid intents:\n" + LABEL_LIST + "\n\n"
    "Reply with the intent name only, exactly as written above, and nothing else."
)

SYSTEM_COT = (
    "You classify messages sent by customers to a bank's support channel into exactly one intent.\n"
    "Valid intents:\n" + LABEL_LIST + "\n\n"
    "First, in one or two sentences, state what the customer wants and which intents are plausible. "
    "Then, on the final line, write `Label: <intent>` using an intent name exactly as written above."
)

SYSTEM_MINIMAL = (
    "Classify the bank customer's message into its intent. Reply with the intent name only."
)

def build_prompt(text, system, examples=()):
    # Chat-formatted prompt; `examples` are (query, intent) pairs inserted as prior turns.
    messages = [{"role": "system", "content": system}]
    for q, intent in examples:
        messages += [{"role": "user", "content": q}, {"role": "assistant", "content": intent}]
    messages.append({"role": "user", "content": text})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

for name, system in [("instruction", SYSTEM_INSTRUCTION), ("cot", SYSTEM_COT), ("minimal", SYSTEM_MINIMAL)]:
    n = len(tokenizer(build_prompt(TEST_TEXTS[0], system), add_special_tokens=False)["input_ids"])
    print(f"{name:<12} prompt tokens: {n}")

print("\n=== INSTRUCTION PROMPT (truncated) ===")
p = build_prompt(TEST_TEXTS[0], SYSTEM_INSTRUCTION)
print(p[:400] + "\n...\n" + p[-250:])

### Mapping model output to an intent

`parse_label` tries, in order:

1. exact match after normalising case and separators (`Card Arrival` → `card_arrival`),
2. the longest intent name contained in the output,
3. a close fuzzy match (`difflib`, cutoff 0.85),

and otherwise returns `INVALID`. For CoT, only the text after the last `Label:` is used (falling back to the last line).

In [ ]:
INVALID = "INVALID"

def normalize(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

NORM2LABEL = {normalize(l): l for l in LABELS}
NORM_KEYS_LONGEST_FIRST = sorted(NORM2LABEL, key=len, reverse=True)

def parse_label(text, cot=False):
    text = text.strip()
    if not text:
        return INVALID
    if cot:
        found = re.findall(r"label\s*:\s*(.+)", text, flags=re.IGNORECASE)
        text = found[-1] if found else text.splitlines()[-1]
    first_line = text.splitlines()[0]

    key = normalize(first_line)
    if key in NORM2LABEL:
        return NORM2LABEL[key]

    padded = f"_{normalize(text)}_"
    for k in NORM_KEYS_LONGEST_FIRST:
        if f"_{k}_" in padded:
            return NORM2LABEL[k]

    close = difflib.get_close_matches(key, list(NORM2LABEL), n=1, cutoff=0.85)
    return NORM2LABEL[close[0]] if close else INVALID

# Self-checks
assert parse_label("card_arrival") == "card_arrival"
assert parse_label("  Card Arrival.\n") == "card_arrival"
assert parse_label("`lost_or_stolen_card`") == "lost_or_stolen_card"
assert parse_label("The intent is card_arrival.") == "card_arrival"
assert parse_label("card_arival") == "card_arrival"
assert parse_label("I don't know") == INVALID
assert parse_label("The user is waiting for a card.\nLabel: card_arrival", cot=True) == "card_arrival"
print("parse_label checks passed")

### Generation, scoring and the results cache

Prompts are sorted by length before batching to reduce padding, and outputs are restored to the original order. Decoding is greedy.

In [ ]:
@torch.no_grad()
def generate(prompts, max_new_tokens, batch_size=GEN_BATCH_SIZE, desc="generating"):
    # Greedy batched generation. Returns (texts, generated-token counts) in input order.
    model.eval()
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    texts, counts = [None] * len(prompts), [0] * len(prompts)
    for start in tqdm(range(0, len(order), batch_size), desc=desc):
        idx = order[start:start + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt",
                        padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
        new = out[:, enc["input_ids"].shape[1]:]
        for i, row in zip(idx, new):
            texts[i]  = tokenizer.decode(row, skip_special_tokens=True).strip()
            counts[i] = int((row != tokenizer.pad_token_id).sum())
    return texts, counts


RESULTS = {}

def bootstrap_ci(preds, golds, n_boot=2000, seed=SEED):
    correct = (np.asarray(preds) == np.asarray(golds)).astype(float)
    rng = np.random.default_rng(seed)
    samples = correct[rng.integers(0, len(correct), size=(n_boot, len(correct)))].mean(axis=1)
    return np.percentile(samples, [2.5, 97.5])

def record(name, preds, raw=None, gen_tokens=None, seconds=None):
    acc = accuracy_score(TEST_GOLD, preds)
    f1  = f1_score(TEST_GOLD, preds, labels=LABELS, average="macro", zero_division=0)
    inv = float(np.mean([p == INVALID for p in preds]))
    lo, hi = bootstrap_ci(preds, TEST_GOLD)
    RESULTS[name] = dict(preds=list(preds), raw=raw, acc=acc, ci=(lo, hi), f1=f1, invalid=inv,
                         gen_tokens=float(np.mean(gen_tokens)) if gen_tokens else None,
                         ms_per_query=1000 * seconds / len(preds) if seconds else None)
    print(f"[{name}] accuracy {acc:.4f} (95% CI {lo:.3f}-{hi:.3f}) | macro-F1 {f1:.4f} | invalid {inv:.1%}")

def run_llm(name, prompts, max_new_tokens, cot=False):
    t0 = time.time()
    texts, counts = generate(prompts, max_new_tokens, desc=name)
    seconds = time.time() - t0
    preds = [parse_label(t, cot=cot) for t in texts]
    record(name, preds, raw=texts, gen_tokens=counts, seconds=seconds)
    return texts, preds

def results_table():
    rows = []
    for name, r in RESULTS.items():
        rows.append({
            "method": name,
            "accuracy": round(r["acc"], 4),
            "95% CI": f"{r['ci'][0]:.3f}-{r['ci'][1]:.3f}",
            "macro-F1": round(r["f1"], 4),
            "invalid": f"{r['invalid']:.1%}",
            "gen tokens/query": None if r["gen_tokens"] is None else round(r["gen_tokens"], 1),
            "ms/query": None if r["ms_per_query"] is None else round(r["ms_per_query"], 1),
        })
    return pd.DataFrame(rows).set_index("method")

def save_results(path="results_banking77.json"):
    serialisable = {k: {**v, "ci": list(v["ci"])} for k, v in RESULTS.items()}
    Path(path).write_text(json.dumps(serialisable, indent=1))

## Baseline: TF-IDF + logistic regression

A cheap reference point, trained on the **same 1,540 queries** the LLM will be fine-tuned on. A second row uses all ~9k training queries to show how much the classical baseline gains from more data.

In [ ]:
def tfidf_baseline(texts, labels):
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1)
    X = vec.fit_transform(texts)
    clf = LogisticRegression(max_iter=3000, C=10)
    clf.fit(X, labels)
    return vec, clf

t0 = time.time()
tfidf_vec, tfidf_clf = tfidf_baseline(list(train_ds["text"]), [id2label[l] for l in train_ds["label"]])
preds = tfidf_clf.predict(tfidf_vec.transform(TEST_TEXTS))
record("TF-IDF + LR (same train)", preds, seconds=time.time() - t0)

vec_all, clf_all = tfidf_baseline(list(train_pool["text"]), [id2label[l] for l in train_pool["label"]])
record("TF-IDF + LR (all train)", clf_all.predict(vec_all.transform(TEST_TEXTS)))

## Act 1: Instruction prompting

The frozen 4-bit model sees the 77 intent names and must return one of them.

In [ ]:
instr_prompts = [build_prompt(t, SYSTEM_INSTRUCTION) for t in TEST_TEXTS]
instr_raw, instr_preds = run_llm("Instruction (zero-shot)", instr_prompts, max_new_tokens=24)

for i in range(3):
    print(f"\nQuery: {TEST_TEXTS[i]}\nGold:  {TEST_GOLD[i]}\nRaw:   {instr_raw[i]!r}\nPred:  {instr_preds[i]}")
save_results()

## Act 2: Chain-of-Thought prompting

The model writes a short rationale before committing to `Label: <intent>`.

**Hypothesis to test:** for short, single-intent queries the decision rests on knowing where the dataset draws boundaries between intents, which reasoning cannot supply. CoT may therefore cost several times more tokens without a matching accuracy gain. Compare the `gen tokens/query` and `ms/query` columns with the accuracy column.

In [ ]:
cot_prompts = [build_prompt(t, SYSTEM_COT) for t in TEST_TEXTS]
cot_raw, cot_preds = run_llm("Chain-of-Thought (zero-shot)", cot_prompts, max_new_tokens=160, cot=True)

for i in range(2):
    print(f"\nQuery: {TEST_TEXTS[i]}\nGold:  {TEST_GOLD[i]}\nPred:  {cot_preds[i]}\nRaw:\n{cot_raw[i]}")
save_results()

### Act 2b (optional): Retrieved few-shot prompting

For each test query, the `FEW_SHOT_K` most similar training queries (TF-IDF cosine similarity) are inserted as prior chat turns. This uses the labelled data without updating any weights, so it isolates how much of fine-tuning's gain could be obtained in-context.

In [ ]:
if RUN_FEW_SHOT:
    train_texts  = list(train_ds["text"])
    train_intent = [id2label[l] for l in train_ds["label"]]
    X_train = tfidf_vec.transform(train_texts)
    X_test  = tfidf_vec.transform(TEST_TEXTS)
    sims = (X_test @ X_train.T).toarray()          # rows are L2-normalised, so this is cosine similarity

    fs_prompts = []
    for i, text in enumerate(TEST_TEXTS):
        nn = np.argsort(-sims[i])[:FEW_SHOT_K][::-1]   # most similar example closest to the query
        fs_prompts.append(build_prompt(text, SYSTEM_INSTRUCTION,
                                       examples=[(train_texts[j], train_intent[j]) for j in nn]))
    run_llm(f"Few-shot retrieved (k={FEW_SHOT_K})", fs_prompts, max_new_tokens=24)
    save_results()

In [ ]:
results_table()

## Act 3: QLoRA fine-tuning

### Prepare the model

`prepare_model_for_kbit_training` casts the remaining non-quantized parameters (norms, embeddings) to fp32 for stable training and enables gradient checkpointing. LoRA adapters of rank `RANK` are then added to every linear projection in attention and the MLP.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=2 * RANK,          # scaling alpha / r = 2
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

total, trainable = count_parameters(model)
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

### Build the supervised fine-tuning dataset

Each example becomes **prompt → completion**, where the completion is the intent name followed by the end-of-turn token. Prompt tokens are masked with `-100`, so only the ~5 completion tokens contribute to the loss.

The prompt is chosen by `TRAIN_PROMPT_STYLE`, and the fine-tuned model is evaluated with the same prompt it was trained on.

In [ ]:
TRAIN_SYSTEM = SYSTEM_MINIMAL if TRAIN_PROMPT_STYLE == "minimal" else SYSTEM_INSTRUCTION
END_OF_TURN = "<|im_end|>" if "<|im_end|>" in tokenizer.get_vocab() else tokenizer.eos_token
MAX_LEN = 1024

def make_sft_example(example):
    prompt = build_prompt(example["text"], TRAIN_SYSTEM)
    completion = id2label[example["label"]] + END_OF_TURN
    prompt_ids     = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + completion_ids
    return {"input_ids": input_ids,
            "labels": [-100] * len(prompt_ids) + completion_ids,
            "attention_mask": [1] * len(input_ids)}

def tokenise_split(ds):
    ds = ds.map(make_sft_example, remove_columns=ds.column_names)
    return ds.filter(lambda x: len(x["input_ids"]) <= MAX_LEN)

train_tok = tokenise_split(train_ds)
eval_tok  = tokenise_split(val_ds)

seq_lens = [len(x) for x in train_tok["input_ids"]]
print(f"Prompt style: {TRAIN_PROMPT_STYLE}")
print(f"Train: {len(train_tok)} | Eval: {len(eval_tok)}")
print(f"Sequence length: mean {np.mean(seq_lens):.0f}, max {max(seq_lens)}")

# Sanity check: the unmasked part must decode to exactly the intent name
ex = train_tok[0]
target = tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])
print(f"Supervised target of example 0: {target!r}")

### Train

Short sequences allow a larger batch. With the full prompt (~600 tokens per sequence) the batch is reduced and gradients are accumulated to keep the effective batch at 16.

In [ ]:
BATCH, ACCUM = (16, 1) if TRAIN_PROMPT_STYLE == "minimal" else (4, 4)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,      # effective batch = 16
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    group_by_length=True,                   # batches of similar length -> less padding
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none",
    optim="paged_adamw_8bit",
    seed=SEED,
)

model.config.use_cache = False              # incompatible with gradient checkpointing during training
trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_tok, eval_dataset=eval_tok, data_collator=data_collator)

t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
model.config.use_cache = True
print(f"Training time: {train_minutes:.1f} min")
if device == "cuda":
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(7, 3.5))
train_log = history.dropna(subset=["loss"])
eval_log  = history.dropna(subset=["eval_loss"])
ax.plot(train_log["step"], train_log["loss"], label="train loss")
ax.plot(eval_log["step"], eval_log["eval_loss"], "o-", label="eval loss")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.legend()
plt.tight_layout()
plt.show()

### Evaluate

Same harness, same test sample, the prompt style used in training.

In [ ]:
qlora_prompts = [build_prompt(t, TRAIN_SYSTEM) for t in TEST_TEXTS]
qlora_name = f"QLoRA (r={RANK}, {TRAIN_PROMPT_STYLE} prompt)"
qlora_raw, qlora_preds = run_llm(qlora_name, qlora_prompts, max_new_tokens=24)
save_results()

In [ ]:
table = results_table()
display(table)

fig, ax = plt.subplots(figsize=(8, 0.5 * len(table) + 1))
names = list(RESULTS)
accs  = [RESULTS[n]["acc"] for n in names]
errs  = np.array([[RESULTS[n]["acc"] - RESULTS[n]["ci"][0], RESULTS[n]["ci"][1] - RESULTS[n]["acc"]]
                  for n in names]).T
ax.barh(names, accs, xerr=errs, capsize=3)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("test accuracy (95% bootstrap CI)")
plt.tight_layout()
plt.show()

**Reading the table.** With 500 test queries the bootstrap interval is roughly ±2-4 points, so differences smaller than that are not reliable; run more seeds or a larger `TEST_SIZE` before drawing conclusions from them. The token and latency columns matter as much as accuracy for deployment.

## Error analysis

All analysis below reuses cached predictions; no further inference is run.

### Per-intent F1 (QLoRA)

In [ ]:
report = classification_report(TEST_GOLD, qlora_preds, labels=LABELS, output_dict=True, zero_division=0)
per_f1 = pd.Series({l: report[l]["f1-score"] for l in LABELS}).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
per_f1.head(10).plot.barh(ax=ax)
ax.set_xlabel("F1")
ax.set_title("10 weakest intents after QLoRA")
plt.tight_layout()
plt.show()

### Most frequent confusions

A full 77×77 matrix is unreadable, so we list the most common (gold → predicted) pairs per method and plot a confusion matrix restricted to the intents involved in QLoRA's top confusions.

In [ ]:
def top_confusions(preds, k=8):
    pairs = collections.Counter((g, p) for g, p in zip(TEST_GOLD, preds) if g != p and p != INVALID)
    return pairs.most_common(k)

for name, r in RESULTS.items():
    print(f"\n{name}")
    for (g, p), n in top_confusions(r["preds"], k=5):
        print(f"  {n:2d}  {g:<38} -> {p}")

In [ ]:
focus = []
for (g, p), _ in top_confusions(qlora_preds, k=8):
    for l in (g, p):
        if l not in focus:
            focus.append(l)

if focus:
    mask = [g in focus for g in TEST_GOLD]
    golds_f = [g for g, m in zip(TEST_GOLD, mask) if m]
    preds_f = [p if p in focus else "(other)" for p, m in zip(qlora_preds, mask) if m]
    cm_labels = focus + ["(other)"]
    cm = confusion_matrix(golds_f, preds_f, labels=cm_labels)[:len(focus)]

    fig, ax = plt.subplots(figsize=(0.55 * len(cm_labels) + 3, 0.55 * len(focus) + 2))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(cm_labels)))
    ax.set_xticklabels(cm_labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(focus)))
    ax.set_yticklabels(focus, fontsize=8)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
    ax.set_xlabel("predicted")
    ax.set_ylabel("gold")
    ax.set_title("QLoRA confusions among the most-confused intents")
    plt.tight_layout()
    plt.show()

### Error types by method

Each error is classified automatically:

* **invalid**: output could not be mapped to an intent,
* **near miss**: the predicted intent is one of the gold intent's 5 nearest neighbours, measured by cosine similarity between TF-IDF centroids of their training queries,
* **far miss**: any other wrong intent.

Near misses point at genuinely overlapping intents; far misses and invalid outputs point at a model that has not learned the label set.

In [ ]:
X_tr = tfidf_vec.transform(list(train_ds["text"]))
y_tr = np.asarray(train_ds["label"])
centroids = np.vstack([np.asarray(X_tr[y_tr == i].mean(axis=0)) for i in range(NUM_LABELS)])
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-12
centroid_sim = centroids @ centroids.T
np.fill_diagonal(centroid_sim, -1)
NEIGHBOURS = {LABELS[i]: {LABELS[j] for j in np.argsort(-centroid_sim[i])[:5]} for i in range(NUM_LABELS)}

def error_type(gold, pred):
    if pred == gold:
        return "correct"
    if pred == INVALID:
        return "invalid"
    return "near miss" if pred in NEIGHBOURS[gold] else "far miss"

breakdown = pd.DataFrame({
    name: collections.Counter(error_type(g, p) for g, p in zip(TEST_GOLD, r["preds"]))
    for name, r in RESULTS.items()
}).T.reindex(columns=["invalid", "far miss", "near miss"]).fillna(0).astype(int)
display(breakdown)

breakdown.plot.barh(stacked=True, figsize=(8, 0.5 * len(breakdown) + 1))
plt.gca().invert_yaxis()
plt.xlabel("errors on test sample")
plt.tight_layout()
plt.show()

### Shared errors

Queries where QLoRA and the TF-IDF baseline are wrong **with the same prediction** are good candidates for ambiguous queries or label noise: two very different models agree against the gold label.

In [ ]:
tfidf_preds = RESULTS["TF-IDF + LR (same train)"]["preds"]
shared = [(t, g, q) for t, g, q, b in zip(TEST_TEXTS, TEST_GOLD, qlora_preds, tfidf_preds)
          if q != g and q == b]
q_wrong = sum(q != g for q, g in zip(qlora_preds, TEST_GOLD))
print(f"QLoRA errors: {q_wrong} | shared with TF-IDF (same wrong label): {len(shared)}\n")
pd.DataFrame(shared[:15], columns=["query", "gold", "both predicted"])

## Save and reload the adapter

Only the LoRA weights are saved; the base model is downloaded separately. We print the real size on disk, then reload the adapter onto a fresh 4-bit base model and check that it reproduces the cached predictions.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
size_mb = sum(f.stat().st_size for f in Path(ADAPTER_DIR).rglob("*.safetensors")) / 1e6
print(f"Adapter weights on disk: {size_mb:.1f} MB")

In [ ]:
del trainer, model
gc.collect()
torch.cuda.empty_cache()

model = PeftModel.from_pretrained(load_base_model(), ADAPTER_DIR)
model.eval()

N_CHECK = 64
texts, _ = generate(qlora_prompts[:N_CHECK], max_new_tokens=24, desc="reload check")
reloaded = [parse_label(t) for t in texts]
agree = np.mean([a == b for a, b in zip(reloaded, qlora_preds[:N_CHECK])])
print(f"Agreement with cached QLoRA predictions on {N_CHECK} queries: {agree:.1%}")

Agreement may be slightly below 100%: gradient-checkpointing and fp32 casts from training are gone after reload, and batched 4-bit kernels are not bit-exact across runs.

**Merging.** `model.merge_and_unload()` folds ΔW = BA back into the base weights so inference no longer needs PEFT. With a 4-bit base this requires dequantizing first; the usual route is to load the base model in 16-bit, attach the adapter, merge, and re-quantize if needed.

## Extensions

* **Learning curve:** retrain with `TRAIN_SIZE` set to about 5, 10 and 20 queries per intent (385, 770, 1540) and plot accuracy against the TF-IDF rows.
* **Prompt style:** set `TRAIN_PROMPT_STYLE = "full"` and compare accuracy, training time and `ms/query` with the minimal prompt.
* **Rank:** compare `RANK` = 4, 16, 64 for accuracy and adapter size.
* **Out-of-scope inputs:** feed queries unrelated to banking and inspect whether the fine-tuned model still emits a confident intent.